In [5]:
!pip install langchain langchain-community langchain-groq
!pip install pymysql sqlalchemy python-dotenv

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_NAME = os.getenv("DB_NAME")

print(f"Groq API Key:{GROQ_API_KEY[-4:]}")
print(f"DB: {DB_USER}|{DB_HOST}:{DB_PORT}|{DB_NAME}")

Groq API Key:czUm
DB: baiastan_8ka|204.168.228.173:3306|gold_market


In [26]:
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    include_tables=[
        "assets", "daily_prices", "macro_indicators",
        "etf_flows", "energy_mining", "market_events",
    ],
    sample_rows_in_table_info=2,
)

print("Connected to MySQL")
print("Tables:", db.get_usable_table_names())

Connected to MySQL
Tables: ['assets', 'daily_prices', 'energy_mining', 'etf_flows', 'macro_indicators', 'market_events']


In [27]:
SYSTEM_PROMPT = """
You are a gold market analyst. You have access to the gold_market MySQL database.
Database schema:
- assets: a reference table of 15 financial assets (gold, silver, S&P 500, ETFs, etc.)
- daily_prices: daily asset prices since 2000 (asset_id FK → assets)
- macro_indicators: macroeconomic data - Fed Rate, Treasury yields, CPI, inflation, real interest rate
- etf_flows: data on GLD and IAU ETFs - prices, signals
- energy_mining: oil, copper, gold production costs, mining margin
- market_events: historical events: crises, wars, monetary policy changes
Rules:
1. Gold is symbol "GC=F" in the assets table
2. To join daily_prices with macro_indicators, use price_date = indicator_date
3. To join daily_prices with energy_mining, use price_date = record_date
4. Always round prices to 2 decimal places and yields to 4 decimal places
"""

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=GROQ_API_KEY)

agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="zero-shot-react-description",
    verbose=True,
    prefix=SYSTEM_PROMPT,
    max_iterations=10,
    handle_parsing_errors=True,
)

print("Done")

Done


In [28]:
def ask(question):
    result = agent.invoke({"input": question})
    print(result["output"])

---
# Prompts

In [ ]:
ask("During which crises did gold prices rise the most? Show the top 5 events by price increase.")

In [29]:
ask("Show the average price of gold, silver and the S&P 500 by year starting in 2010")



> Entering new SQL Agent Executor chain...
Thought: I should first list the tables in the database to see what I can query.
Action: sql_db_list_tables
Action Input: assets, daily_prices, energy_mining, etf_flows, macro_indicators, market_eventsThought: I now know the tables in the database. I can see that the daily_prices table seems to be the most relevant for this query, as it contains the daily prices of the assets. I should query the schema of this table to see what fields are available.
Action: sql_db_schema
Action Input: daily_prices
CREATE TABLE daily_prices (
	price_id INTEGER NOT NULL AUTO_INCREMENT, 
	asset_id INTEGER NOT NULL, 
	price_date DATE NOT NULL, 
	close_price DECIMAL(15, 4), 
	PRIMARY KEY (price_id), 
	CONSTRAINT daily_prices_ibfk_1 FOREIGN KEY(asset_id) REFERENCES assets (asset_id)
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_unicode_ci ENGINE=InnoDB

/*
2 rows from daily_prices table:
price_id	asset_id	price_date	close_price
1	5	2000-01-03	100.2200
2	6	2000-01-03	14

In [ ]:
ask("What is the average price of gold when the real interest rate is negative (real_rate_calc < 0)?")

In [ ]:
ask("")